In [1]:
%useLatestDescriptors
%use kstats
%use kandy
%use dataframe

import kotlin.random.Random
import kotlin.math.abs
import java.util.Locale

fun Double.fmt(d: Int) = String.format(Locale.US, "%.${d}f", this)

We analyze two real-world datasets:
- **E-commerce Landing Page** ([Kaggle](https://www.kaggle.com/datasets/zhangluyuan/ab-testing))
  — ~294K users randomly assigned to old vs new checkout page, binary outcome (converted / not)
- **Marketing Campaign** ([Kaggle](https://www.kaggle.com/datasets/amirmotefaker/ab-testing-dataset))
  — daily campaign metrics (spend, clicks, impressions, purchases) for control vs test campaign

**Structure**:
- **Act 1** — Binary metric: conversion rate (proportion z-test, chi-squared, Bayesian)
- **Act 2** — Rate metrics: CTR, purchase rate (paired t-test, Wilcoxon, effect sizes, correlation)

## Which Test to Use?

Use this reference card to decide which tools to apply at each step.

| Metric Type | Test | Effect Size | CI Method |
|-------------|------|-------------|-----------|
| Binary (converted / not) | `proportionZTest()` | Cohen's h (`cohensH()`) | Wilson (`binomialTest(ciMethod = CIMethod.WILSON)`) |
| Continuous, normal (independent) | `tTest()` | Cohen's d (`cohensD()`) / Hedges' g (`hedgesG()`) | From t-test result |
| Continuous, normal (paired) | `pairedTTest()` | Paired dz (manual: mean/SD of diffs) | From t-test result |
| Continuous, non-normal | `mannWhitneyUTest()` / `wilcoxonSignedRankTest()` | Rank-biserial r / Cliff's δ (not yet in kstats) | Bootstrap (out of scope) |
| Contingency table | `chiSquaredIndependenceTest()` | — | — |
| Multiple metrics | Apply correction: `bonferroniCorrection()`, `holmBonferroniCorrection()`, `benjaminiHochbergCorrection()` | | |

## ACT 1: Binary Metric — Conversion Rate

### 1. Data Loading & Cleaning

**Dataset**: [Udacity A/B Testing — E-commerce Landing Page](https://www.kaggle.com/datasets/zhangluyuan/ab-testing)
— ~294K users randomly assigned to control (old page) or treatment (new page), binary outcome (converted / not).

In [2]:
val df = DataFrame.readCsv("data/landing-page-ab-test.csv")
df.head()

user_id,timestamp,group,landing_page,converted
851104,2017-01-21T22:11:48.556739,control,old_page,0
804228,2017-01-12T08:01:45.159739,control,old_page,0
661590,2017-01-11T16:55:06.154213,treatment,new_page,0
853541,2017-01-08T18:28:03.143765,treatment,new_page,0
864975,2017-01-21T01:52:26.210827,control,old_page,1


In [3]:
println("Shape: ${df.rowsCount()} rows × ${df.columnsCount()} columns")
df.describe()

Shape: 294478 rows × 5 columns


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
user_id,Int,294478,290584,0,767017,2,"787974,124733","91210,823776",630000,"709031,916667","787933,500000","866912,083333",945999
timestamp,LocalDateTime,294478,294478,0,2017-01-21T22:11:48.556739,1,null,null,2017-01-02T13:42:05.378582,2017-01-08T02:06:47.599494,2017-01-13T13:21:03.498370,2017-01-19T01:43:49.552409,2017-01-24T13:41:54.460509
group,String,294478,2,0,treatment,147276,null,null,control,control,treatment,treatment,treatment
landing_page,String,294478,2,0,old_page,147239,null,null,new_page,new_page,new_page,old_page,old_page
converted,Int,294478,2,0,0,259241,"0,119659","0,324563",0,"0,000000","0,000000","0,000000",1


In [4]:
// Check for duplicate user_ids
val duplicateUsers = df.groupBy("user_id").count().filter { "count"<Int>() > 1 }
println("Duplicate user_ids: ${duplicateUsers.rowsCount()}")

// Check for mismatched rows (treatment + old_page or control + new_page)
val mismatched = df.filter {
    ("group"<String>() == "treatment" && "landing_page"<String>() == "old_page") ||
    ("group"<String>() == "control" && "landing_page"<String>() == "new_page")
}
"Mismatched rows: ${mismatched.rowsCount()}"

Duplicate user_ids: 3894


Mismatched rows: 3893

In [5]:
// Remove mismatched rows, then deduplicate by user_id (keep first)
val clean = df
    .filter {
        !("group"<String>() == "treatment" && "landing_page"<String>() == "old_page") &&
        !("group"<String>() == "control" && "landing_page"<String>() == "new_page")
    }
    .distinctBy("user_id")

"After cleaning: ${clean.rowsCount()} rows (removed ${df.rowsCount() - clean.rowsCount()})"

After cleaning: 290584 rows (removed 3894)

In [6]:
val controlGroup = clean.filter { "group"<String>() == "control" }
val treatmentGroup = clean.filter { "group"<String>() == "treatment" }

val controlConverted = controlGroup.filter { "converted"<Int>() == 1 }.rowsCount()
val controlTotal = controlGroup.rowsCount()
val treatmentConverted = treatmentGroup.filter { "converted"<Int>() == 1 }.rowsCount()
val treatmentTotal = treatmentGroup.rowsCount()

val controlRate = controlConverted.toDouble() / controlTotal
val treatmentRate = treatmentConverted.toDouble() / treatmentTotal

println("Control:   $controlConverted / $controlTotal = ${controlRate.fmt(4)} (${(controlRate * 100).fmt(2)}%)")
println("Treatment: $treatmentConverted / $treatmentTotal = ${treatmentRate.fmt(4)} (${(treatmentRate * 100).fmt(2)}%)")
"Δ = ${(treatmentRate - controlRate).fmt(4)} (${((treatmentRate - controlRate) * 100).fmt(2)} pp)"

Control:   17489 / 145274 = 0.1204 (12.04%)
Treatment: 17264 / 145310 = 0.1188 (11.88%)


Δ = -0.0016 (-0.16 pp)

In [7]:
val groups = listOf("Control\n${(controlRate * 100).fmt(2)}%", "Treatment\n${(treatmentRate * 100).fmt(2)}%")
val rates = listOf(controlRate * 100, treatmentRate * 100)

plot {
    bars {
        x(groups); y(rates)
        fillColor = Color.hex("#4dabf7")
        alpha = 0.8
    }
    layout {
        title = "Conversion Rate by Group (%)"
        size = 500 to 400
        xAxisLabel = "Group"
        yAxisLabel = "Conversion Rate (%)"
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="8byFLF"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 500.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("8byFLF");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Conversion Rate by Group (%)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Group"
},
"y":{
"title":"Conversion Rate (%)"
}
},
"data":{
},
"ggsize":{
"width":500.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["Control\n12.04%","Treatment\n11.88%"],
"y":[12.03863045004612,11.880806551510565]
},
"sampling":"none",
"alpha":0.8,
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"fill":"#4dabf7",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"2"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Control 
 
 
 12.04% 
 
 
 
 
 
 
 
 
 Treatment 
 
 
 11.88% 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 12 
 
 
 
 
 
 
 
 
 Conversion Rate by Group (%) 
 
 
 
 
 Conversion Rate (%) 
 
 
 
 
 Group

The difference is tiny: ~0.16 percentage points. The question is whether this is noise or a real effect.

### 2. SRM Check — Sample Ratio Mismatch

Before any hypothesis test, verify that randomization worked correctly. If the split ratio deviates
from 50/50 more than chance would allow, the experiment is compromised — all downstream results
are untrustworthy.

**Sample Ratio Mismatch (SRM)** is checked with a one-sample proportion z-test: is the observed
split consistent with a 50/50 ratio?

In [8]:
val srmTest = proportionZTest(
    successes = controlTotal,
    trials = controlTotal + treatmentTotal,
    p0 = 0.5
)

println("SRM Check (proportion z-test)")
println("  Control:   $controlTotal users")
println("  Treatment: $treatmentTotal users")
println("  Ratio:     ${(controlTotal.toDouble() / (controlTotal + treatmentTotal)).fmt(4)}")
println("  p-value:   ${srmTest.pValue.fmt(4)}")
"  SRM detected: ${srmTest.isSignificant()}"

SRM Check (proportion z-test)
  Control:   145274 users
  Treatment: 145310 users
  Ratio:     0.4999
  p-value:   0.9468


  SRM detected: false

p >> 0.05 — no SRM detected. The 50/50 split is intact and randomization looks correct.

### 3. Power Analysis — Pre-Experiment Planning

Power analysis should happen **before** data collection to determine the required sample size.
We ask: "How many users do we need to detect a 1 percentage point lift from a 12% baseline with
80% power at α = 0.05?"

This ensures the experiment is neither underpowered (missing real effects) nor wasteful (running
longer than necessary).

In [9]:
// Effect size for a 1 pp lift: 12% → 13%
val h = cohensH(p1 = 0.13, p2 = 0.12)
println("Cohen's h for 12% → 13%: ${h.fmt(4)}")

val requiredN = proportionZTestRequiredN(effectSize = h, power = 0.8)
println("Required per group: $requiredN users")
"Total: ${requiredN * 2} users"

Cohen's h for 12% → 13%: 0.0302
Required per group: 17164 users


Total: 34328 users

In [10]:
// With ~145K per group, what power do we actually have?
val actualPower = proportionZTestPower(effectSize = h, n = 145_000)
println("Power with N=145K per group: ${(actualPower * 100).fmt(1)}%")

val mde = proportionZTestMinimumEffect(n = 145_000, power = 0.8)
"MDE (Cohen's h): ${mde.fmt(4)}"

Power with N=145K per group: 100.0%


MDE (Cohen's h): 0.0104

In [11]:
val powers = listOf(0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99)
val sizes = powers.map { p -> proportionZTestRequiredN(effectSize = h, power = p) }

plot {
    line {
        x(powers.map { it * 100 }); y(sizes.map { it.toDouble() })
        color = Color.hex("#1971c2")
        width = 2.0
    }
    points {
        x(powers.map { it * 100 }); y(sizes.map { it.toDouble() })
        color = Color.hex("#1971c2")
        size = 5.0
    }
    layout {
        title = "Required N per Group vs Power (detecting 1 pp lift from 12%)"
        size = 700 to 400
        xAxisLabel = "Power (%)"
        yAxisLabel = "Required N per Group"
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="lDFmwx"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("lDFmwx");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Required N per Group vs Power (detecting 1 pp lift from 12%)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Power (%)"
},
"y":{
"title":"Required N per Group"
}
},
"data":{
},
"ggsize":{
"width":700.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[50.0,60.0,70.0,80.0,90.0,95.0,99.0],
"y":[8401.0,10713.0,13497.0,17164.0,22977.0,28416.0,40175.0]
},
"color":"#1971c2",
"size":2.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[50.0,60.0,70.0,80.0,90.0,95.0,99.0],
"y":[8401.0,10713.0,13497.0,17164.0,22977.0,28416.0,40175.0]
},
"color":"#1971c2",
"size":5.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"5"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 55 
 
 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 
 
 65 
 
 
 
 
 
 
 
 
 70 
 
 
 
 
 
 
 
 
 75 
 
 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 
 
 85 
 
 
 
 
 
 
 
 
 90 
 
 
 
 
 
 
 
 
 95 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 
 
 10,000 
 
 
 
 
 
 
 15,000 
 
 
 
 
 
 
 20,000 
 
 
 
 
 
 
 25,000 
 
 
 
 
 
 
 30,000 
 
 
 
 
 
 
 35,000 
 
 
 
 
 
 
 40,000 
 
 
 
 
 
 
 
 
 Required N per Group vs Power (detecting 1 pp lift from 12%) 
 
 
 
 
 Required N per Group 
 
 
 
 
 Power (%)

With ~145K per group we have >99% power to detect a 1 pp lift. The experiment is well-powered.

### 4. Primary Test — Two-Sample Proportion z-Test

The core question: **is the conversion rate different between control and treatment?**

For comparing two binomial proportions on large samples, the proportion z-test is the standard method.
It compares the observed difference against sampling variability under the null hypothesis of equal rates.

In [12]:
val conversionTest = proportionZTest(
    successes1 = treatmentConverted, trials1 = treatmentTotal,
    successes2 = controlConverted,   trials2 = controlTotal
)
val ci = conversionTest.confidenceInterval!!

println("Two-Sample Proportion z-Test")
println("  z-statistic: ${conversionTest.statistic.fmt(4)}")
println("  p-value:     ${conversionTest.pValue.fmt(4)}")
println("  95% CI for Δ(p): [${ci.lower.fmt(4)}, ${ci.upper.fmt(4)}]")
"  Significant at α=0.05: ${conversionTest.isSignificant()}"

Two-Sample Proportion z-Test
  z-statistic: -1.3109
  p-value:     0.1899
  95% CI for Δ(p): [-0.0039, 0.0008]


  Significant at α=0.05: false

p > 0.05 — we **cannot reject** the null hypothesis. The new page does not significantly change conversion rate.

The 95% CI for the difference includes zero, consistent with no effect.

#### One-sided check

The company only cares if the new page is **better**. A one-sided test has more power
to detect an improvement, but p ≈ 0.905 here strongly suggests the treatment is actually
*worse* (or at best equal) — this is not a borderline result.

In [13]:
val oneSided = proportionZTest(
    successes1 = treatmentConverted, trials1 = treatmentTotal,
    successes2 = controlConverted,   trials2 = controlTotal,
    alternative = Alternative.GREATER
)
println("One-sided (treatment > control) p = ${oneSided.pValue.fmt(4)}")
"Significant: ${oneSided.isSignificant()}"

One-sided (treatment > control) p = 0.9051


Significant: false

### 5. Effect Size — Cohen's h

p-value answers "is there a difference?". Effect size answers "**how big** is the difference?"

With N=290K even tiny differences can become "significant". Cohen's h puts the difference on a
standardized scale independent of sample size.

| Cohen's h | Interpretation |
|-----------|----------------|
| < 0.2 | Negligible |
| 0.2 | Small |
| 0.5 | Medium |
| 0.8 | Large |

In [14]:
val effectH = cohensH(p1 = treatmentRate, p2 = controlRate)
println("Cohen's h = ${effectH.fmt(4)}")
"Interpretation: negligible (|h| < 0.2)"

Cohen's h = -0.0049


Interpretation: negligible (|h| < 0.2)

In [15]:
// Context: Cohen's h for various conversion lifts from a 12% baseline
val lifts = listOf(0.005, 0.01, 0.02, 0.03, 0.05, 0.10)
val hValues = lifts.map { lift -> cohensH(p1 = 0.12 + lift, p2 = 0.12) }
val liftLabels = lifts.map { "+${(it * 100).fmt(1)} pp" }

plot {
    bars {
        x(liftLabels); y(hValues)
        fillColor = Color.hex("#9775fa")
        alpha = 0.8
    }
    line {
        x(listOf(liftLabels.first(), liftLabels.last()))
        y(listOf(0.2, 0.2))
        color = Color.hex("#e03131")
        width = 1.5
    }
    layout {
        title = "Cohen's h by Conversion Lift from 12% Baseline (red = small effect threshold)"
        size = 700 to 400
        xAxisLabel = "Lift (percentage points)"
        yAxisLabel = "Cohen's h"
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="E5Lcw4"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("E5Lcw4");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Cohen's h by Conversion Lift from 12% Baseline (red = small effect threshold)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Lift (percentage points)"
},
"y":{
"title":"Cohen's h"
}
},
"data":{
},
"ggsize":{
"width":700.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["+0.5 pp","+1.0 pp","+2.0 pp","+3.0 pp","+5.0 pp","+10.0 pp"],
"y":[0.01525103603407274,0.03024275667390597,0.05951079608252363,0.08791561840480067,0.14249435414546396,0.2689273150144915]
},
"sampling":"none",
"alpha":0.8,
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"fill":"#9775fa",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["+0.5 pp","+10.0 pp"],
"y":[0.2,0.2]
},
"color":"#e03131",
"size":1.5,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"8"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 +0.5 pp 
 
 
 
 
 
 
 
 
 +1.0 pp 
 
 
 
 
 
 
 
 
 +2.0 pp 
 
 
 
 
 
 
 
 
 +3.0 pp 
 
 
 
 
 
 
 
 
 +5.0 pp 
 
 
 
 
 
 
 
 
 +10.0 pp 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 0.05 
 
 
 
 
 
 
 0.1 
 
 
 
 
 
 
 0.15 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.25 
 
 
 
 
 
 
 
 
 Cohen's h by Conversion Lift from 12% Baseline (red = small effect threshold) 
 
 
 
 
 Cohen's h 
 
 
 
 
 Lift (percentage points)

Even a 2 pp lift from a 12% baseline is a small effect by Cohen's standards. In e-commerce A/B testing, effects are almost always small — that's why large sample sizes are essential.

### 6. Confidence Intervals — Wilson Score

The Wald CI from the z-test can misbehave near 0 or 1. The **Wilson score interval** is recommended
for per-group conversion rate estimates, especially with small samples.

For small samples, Wilson and Agresti-Coull can differ significantly from Clopper-Pearson. At N=145K
they converge — we show Wilson as the best default choice.

In [16]:
val controlCI = binomialTest(successes = controlConverted, trials = controlTotal, ciMethod = CIMethod.WILSON)
val treatmentCI = binomialTest(successes = treatmentConverted, trials = treatmentTotal, ciMethod = CIMethod.WILSON)

println("Control   Wilson 95% CI: [${controlCI.confidenceInterval!!.lower.fmt(4)}, ${controlCI.confidenceInterval!!.upper.fmt(4)}]")
"Treatment Wilson 95% CI: [${treatmentCI.confidenceInterval!!.lower.fmt(4)}, ${treatmentCI.confidenceInterval!!.upper.fmt(4)}]"

Control   Wilson 95% CI: [0.1187, 0.1221]


Treatment Wilson 95% CI: [0.1172, 0.1205]

### 7. Robustness Check — Chi-Squared Test

The same hypothesis can be tested with a 2×2 contingency table. For large N the chi-squared
statistic equals z² — a useful sanity check.

In [17]:
val table = arrayOf(
    intArrayOf(controlConverted, controlTotal - controlConverted),
    intArrayOf(treatmentConverted, treatmentTotal - treatmentConverted)
)
val chiResult = chiSquaredIndependenceTest(table)

println("Chi-squared: χ²=${chiResult.statistic.fmt(4)}, p=${chiResult.pValue.fmt(4)}, df=${chiResult.degreesOfFreedom}")
println("Significant: ${chiResult.isSignificant()}")
"Consistency check: z²=${(conversionTest.statistic * conversionTest.statistic).fmt(4)}, χ²=${chiResult.statistic.fmt(4)}"

Chi-squared: χ²=1.7185, p=0.1899, df=1.0
Significant: false


Consistency check: z²=1.7185, χ²=1.7185

Both tests agree: no significant difference. z² ≈ χ² confirms internal consistency.

### 8. Bayesian A/B Testing

The frequentist approach gives a binary answer: reject or don't reject. The Bayesian approach
answers a more intuitive question: **what is the probability that the treatment is better?**

We use the Beta-Binomial conjugate model:
- **Prior**: Beta(1, 1) — uninformative (uniform on [0, 1])
- **Posterior**: Beta(1 + successes, 1 + failures) — updated belief after seeing data
- **Decision**: Compare posterior distributions via Monte Carlo sampling

In [18]:
val posteriorControl = BetaDistribution(
    alpha = 1.0 + controlConverted,
    beta = 1.0 + (controlTotal - controlConverted)
)
val posteriorTreatment = BetaDistribution(
    alpha = 1.0 + treatmentConverted,
    beta = 1.0 + (treatmentTotal - treatmentConverted)
)

println("Posterior Control:   Beta(${1 + controlConverted}, ${1 + controlTotal - controlConverted}), mean=${posteriorControl.mean.fmt(6)}")
"Posterior Treatment: Beta(${1 + treatmentConverted}, ${1 + treatmentTotal - treatmentConverted}), mean=${posteriorTreatment.mean.fmt(6)}"

Posterior Control:   Beta(17490, 127786), mean=0.120392


Posterior Treatment: Beta(17265, 128047), mean=0.118813

In [19]:
// Monte Carlo: P(treatment > control)
val rng = Random(42)
val nSamples = 100_000
val samplesControl = posteriorControl.sample(nSamples, rng)
val samplesTreatment = posteriorTreatment.sample(nSamples, rng)

val pTreatmentBetter = (0 until nSamples).count {
    samplesTreatment[it] > samplesControl[it]
} / nSamples.toDouble()

println("P(treatment > control) = ${pTreatmentBetter.fmt(4)} (${(pTreatmentBetter * 100).fmt(1)}%)")
"P(control > treatment) = ${(1.0 - pTreatmentBetter).fmt(4)} (${((1.0 - pTreatmentBetter) * 100).fmt(1)}%)"

P(treatment > control) = 0.0949 (9.5%)


P(control > treatment) = 0.9051 (90.5%)

In [20]:
// Density overlay: posterior distributions
val xMin = 0.1170
val xMax = 0.1220
val nPoints = 500
val xGrid = DoubleArray(nPoints) { xMin + (xMax - xMin) * it / (nPoints - 1) }
val pdfControl = xGrid.map { posteriorControl.pdf(it) }
val pdfTreatment = xGrid.map { posteriorTreatment.pdf(it) }

plot {
    line {
        x(xGrid.toList()); y(pdfControl)
        color = Color.hex("#4dabf7")
        width = 2.0
    }
    line {
        x(xGrid.toList()); y(pdfTreatment)
        color = Color.hex("#ff8787")
        width = 2.0
    }
    layout {
        title = "Posterior Distributions: Control (blue) vs Treatment (red)"
        size = 800 to 400
        xAxisLabel = "Conversion Rate"
        yAxisLabel = "Density"
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="2IBDsz"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 800.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("2IBDsz");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Posterior Distributions: Control (blue) vs Treatment (red)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Conversion Rate"
},
"y":{
"title":"Density"
}
},
"data":{
},
"ggsize":{
"width":800.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[0.117,0.11701002004008017,0.11702004008016033,0.11703006012024049,0.11704008016032065,0.11705010020040081,0.11706012024048097,0.11707014028056113,0.1170801603206413,0.11709018036072145,0.11710020040080162,0.11711022044088178,0.11712024048096192,0.11713026052104208,0.11714028056112225,0.1171503006012024,0.11716032064128257,0.11717034068136273,0.11718036072144289,0.11719038076152305,0.11720040080160321,0.11721042084168337,0.11722044088176353,0.1172304609218437,0.11724048096192385,0.11725050100200402,0.11726052104208418,0.11727054108216434,0.1172805611222445,0.11729058116232466,0.11730060120240482,0.11731062124248498,0.11732064128256514,0.1173306613226453,0.11734068136272546,0.11735070140280562,0.11736072144288577,0.11737074148296593,0.11738076152304609,0.11739078156312625,0.11740080160320641,0.11741082164328658,0.11742084168336674,0.1174308617234469,0.11744088176352706,0.11745090180360722,0.11746092184368738,0.11747094188376754,0.1174809619238477,0.11749098196392786,0.11750100200400802,0.11751102204408818,0.11752104208416835,0.1175310621242485,0.11754108216432867,0.11755110220440883,0.11756112224448899,0.11757114228456915,0.11758116232464931,0.11759118236472947,0.11760120240480962,0.11761122244488978,0.11762124248496994,0.1176312625250501,0.11764128256513026,0.11765130260521042,0.11766132264529058,0.11767134268537074,0.1176813627254509,0.11769138276553107,0.11770140280561123,0.11771142284569139,0.11772144288577155,0.11773146292585171,0.11774148296593187,0.11775150300601203,0.11776152304609219,0.11777154308617235,0.11778156312625251,0.11779158316633268,0.11780160320641284,0.117811623246493,0.11782164328657316,0.11783166332665332,0.11784168336673347,0.11785170340681363,0.11786172344689379,0.11787174348697395,0.11788176352705411,0.11789178356713427,0.11790180360721443,0.11791182364729459,0.11792184368737475,0.11793186372745491,0.11794188376753507,0.11795190380761524,0.1179619238476954,0.11797194388777556,0.11798196392785572,0.11799198396793588,0.11800200400801604,0.1180120240480962,0.11802204408817636,0.11803206412825652,0.11804208416833668,0.11805210420841684,0.118062124248497,0.11807214428857715,0.11808216432865731,0.11809218436873747,0.11810220440881763,0.1181122244488978,0.11812224448897796,0.11813226452905812,0.11814228456913828,0.11815230460921844,0.1181623246492986,0.11817234468937876,0.11818236472945892,0.11819238476953908,0.11820240480961924,0.1182124248496994,0.11822244488977957,0.11823246492985973,0.11824248496993989,0.11825250501002005,0.11826252505010021,0.11827254509018037,0.11828256513026053,0.11829258517034069,0.11

In [21]:
// 95% Credible intervals
println("Control   95% CI: [${posteriorControl.quantile(0.025).fmt(5)}, ${posteriorControl.quantile(0.975).fmt(5)}]")
println("Treatment 95% CI: [${posteriorTreatment.quantile(0.025).fmt(5)}, ${posteriorTreatment.quantile(0.975).fmt(5)}]")

val diffs = DoubleArray(nSamples) { samplesTreatment[it] - samplesControl[it] }
diffs.sort()
val diffLow = diffs[(nSamples * 0.025).toInt()]
val diffHigh = diffs[(nSamples * 0.975).toInt()]
println("\nDifference 95% CI: [${diffLow.fmt(5)}, ${diffHigh.fmt(5)}]")
"Contains zero: ${diffLow <= 0.0 && diffHigh >= 0.0}"

Control   95% CI: [0.11872, 0.12207]
Treatment 95% CI: [0.11715, 0.12048]

Difference 95% CI: [-0.00395, 0.00078]


Contains zero: true

The Bayesian approach tells the same story: P(treatment > control) is only ~9.5%. Most
organizations require P(B > A) > 95% to ship a change.

**Frequentist vs Bayesian**:
- Frequentist: "We cannot reject H₀" — binary decision, p-value depends on sample size
- Bayesian: "There's a ~9.5% chance the new page is better" — direct probability statement,
  more intuitive for stakeholders

Both agree here: there is no compelling evidence to launch the new page.

> **Note on priors**: We used a uniform Beta(1,1) prior. With N=145K observations, the prior
> has virtually no effect on the posterior. For smaller experiments (N < 1000), the choice of
> prior matters more — consider using an informative prior based on historical conversion rates.

---

## ACT 2: Continuous Metrics — Marketing Campaign

We now switch to **continuous metrics** where different statistical tools are needed.

**Dataset**: [Marketing Campaign A/B Testing](https://www.kaggle.com/datasets/amirmotefaker/ab-testing-dataset)
— daily campaign metrics (spend, clicks, impressions, purchases) for control vs test campaign.

In [22]:
val dfControlRaw = DataFrame.readCsv("data/campaign-control.csv", delimiter = ';')
val dfTestRaw = DataFrame.readCsv("data/campaign-test.csv", delimiter = ';')

// Both CSVs share the same 30 dates. Drop nulls independently, then keep only matched dates.
val dfControlClean = dfControlRaw.dropNulls()
val dfTestClean = dfTestRaw.dropNulls()

val ctrlDates = dfControlClean["Date"].toList().map { it.toString() }.toSet()
val testDates = dfTestClean["Date"].toList().map { it.toString() }.toSet()
val pairedDates = ctrlDates.intersect(testDates)

// Filter to matched dates and sort by Date to ensure paired alignment
val dfControlPaired = dfControlClean.filter { "Date"<Any>().toString() in pairedDates }.sortBy("Date")
val dfTestPaired = dfTestClean.filter { "Date"<Any>().toString() in pairedDates }.sortBy("Date")

println("Matched days: ${dfControlPaired.rowsCount()} (dropped ${30 - dfControlPaired.rowsCount()} days with missing data)")
println("Control: ${dfControlPaired.rowsCount()} rows × ${dfControlPaired.columnsCount()} columns")
println("Test:    ${dfTestPaired.rowsCount()} rows × ${dfTestPaired.columnsCount()} columns")
dfControlPaired.head()

Matched days: 29 (dropped 1 days with missing data)
Control: 29 rows × 10 columns
Test:    29 rows × 10 columns


Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase
Control Campaign,"1082019,000000",2280,82702,56930,7016,2290,2159,1819,618
Control Campaign,"2082019,000000",1757,121040,102513,8110,2033,1841,1219,511
Control Campaign,"3082019,000000",2343,131711,110862,6508,1737,1549,1134,372
Control Campaign,"4082019,000000",1940,72878,61235,3065,1042,982,1183,340
Control Campaign,"6082019,000000",3083,109076,87998,4028,1709,1249,784,764


In [23]:
dfControlPaired.columnNames()

[Campaign Name, Date, Spend [USD], # of Impressions, Reach, # of Website Clicks, # of Searches, # of View Content, # of Add to Cart, # of Purchase]

In [24]:
// Raw counts (kept for correlation heatmap)
val controlPurchase = dfControlPaired["# of Purchase"].toList().map { (it as Number).toDouble() }.toDoubleArray()
val treatmentPurchase = dfTestPaired["# of Purchase"].toList().map { (it as Number).toDouble() }.toDoubleArray()
val controlClicks = dfControlPaired["# of Website Clicks"].toList().map { (it as Number).toDouble() }.toDoubleArray()
val treatmentClicks = dfTestPaired["# of Website Clicks"].toList().map { (it as Number).toDouble() }.toDoubleArray()
val controlImpressions = dfControlPaired["# of Impressions"].toList().map { (it as Number).toDouble() }.toDoubleArray()
val treatmentImpressions = dfTestPaired["# of Impressions"].toList().map { (it as Number).toDouble() }.toDoubleArray()

// Rate metrics: normalize by daily impressions
val controlCTR = controlClicks.zip(controlImpressions).map { (c, i) -> c / i }.toDoubleArray()
val treatmentCTR = treatmentClicks.zip(treatmentImpressions).map { (c, i) -> c / i }.toDoubleArray()
val controlPurchaseRate = controlPurchase.zip(controlImpressions).map { (p, i) -> p / i }.toDoubleArray()
val treatmentPurchaseRate = treatmentPurchase.zip(treatmentImpressions).map { (p, i) -> p / i }.toDoubleArray()

// Exposure comparison
val ctrlImprTotal = controlImpressions.sum()
val treatImprTotal = treatmentImpressions.sum()
val ctrlSpendTotal = dfControlPaired["Spend [USD]"].toList().sumOf { (it as Number).toDouble() }
val treatSpendTotal = dfTestPaired["Spend [USD]"].toList().sumOf { (it as Number).toDouble() }
println("Exposure comparison:")
println("  Impressions — Control: ${ctrlImprTotal.fmt(0)}, Treatment: ${treatImprTotal.fmt(0)} (${((treatImprTotal / ctrlImprTotal - 1) * 100).fmt(1)}%)")
println("  Spend       — Control: ${ctrlSpendTotal.fmt(0)}, Treatment: ${treatSpendTotal.fmt(0)} (${((treatSpendTotal / ctrlSpendTotal - 1) * 100).fmt(1)}%)")
println()

val ctrlCTRDesc = controlCTR.describe()
val treatCTRDesc = treatmentCTR.describe()
val ctrlPurchRateDesc = controlPurchaseRate.describe()
val treatPurchRateDesc = treatmentPurchaseRate.describe()

println("CTR (Control):           mean=${(ctrlCTRDesc.mean * 100).fmt(2)}%, sd=${(ctrlCTRDesc.standardDeviation * 100).fmt(2)}%")
println("CTR (Treatment):         mean=${(treatCTRDesc.mean * 100).fmt(2)}%, sd=${(treatCTRDesc.standardDeviation * 100).fmt(2)}%")
println("Purchase Rate (Control): mean=${(ctrlPurchRateDesc.mean * 100).fmt(3)}%, sd=${(ctrlPurchRateDesc.standardDeviation * 100).fmt(3)}%")
"Purchase Rate (Treatment): mean=${(treatPurchRateDesc.mean * 100).fmt(3)}%, sd=${(treatPurchRateDesc.standardDeviation * 100).fmt(3)}%"

Exposure comparison:
  Impressions — Control: 3177233, Treatment: 2123249 (-33.2%)
  Spend       — Control: 66818, Treatment: 74595 (11.6%)

CTR (Control):           mean=5.10%, sd=2.05%
CTR (Treatment):         mean=10.42%, sd=6.82%
Purchase Rate (Control): mean=0.500%, sd=0.217%


Purchase Rate (Treatment): mean=0.848%, sd=0.530%

The marketing campaign ran for 30 days. Each row is one day of metrics. Both campaigns ran on
the same calendar days, making this a **paired (repeated-measures) design** — day-level factors
(weekday effects, seasonality, market conditions) affect both groups simultaneously.

**Why rates, not raw counts?** The two groups have very different exposure: treatment received
~30% fewer impressions but ~12% more spend. Comparing raw daily counts (clicks, purchases)
would conflate the campaign effect with the exposure difference. Instead, we normalize by
daily impressions to get **CTR** (clicks / impressions) and **purchase rate** (purchases / impressions).

**Why paired tests?** Since control and test observations are matched by date, a paired t-test
(or Wilcoxon signed-rank) removes day-to-day variability, increasing power compared to an
independent-samples test.

### 9. EDA — Paired Differences (ΔCTR & ΔPurchase Rate)

In [25]:
// Delta plot: paired differences (Treatment − Control) per day
val ctrDiffsPct = DoubleArray(treatmentCTR.size) { (treatmentCTR[it] - controlCTR[it]) * 100 }
val ctrDiffMean = ctrDiffsPct.mean()
val ctrDiffSE = sqrt(ctrDiffsPct.variance() / ctrDiffsPct.size)
val ctrCI95 = 2.048 * ctrDiffSE  // t_{0.025, df=28}

val rng1 = kotlin.random.Random(42)

plot {
    // Zero reference line
    line {
        x(listOf(0.0, 0.0))
        y(listOf(-0.5, 0.5))
        color = Color.hex("#adb5bd")
        width = 1.0
    }
    // Individual day differences (jittered strip)
    points {
        x(ctrDiffsPct.toList())
        y(List(ctrDiffsPct.size) { rng1.nextDouble(-0.35, 0.35) })
        color = Color.hex("#4dabf7")
        size = 4.5
        alpha = 0.7
    }
    // 95% CI bar
    line {
        x(listOf(ctrDiffMean - ctrCI95, ctrDiffMean + ctrCI95))
        y(listOf(0.0, 0.0))
        color = Color.hex("#e03131")
        width = 3.0
    }
    // Mean point
    points {
        x(listOf(ctrDiffMean))
        y(listOf(0.0))
        color = Color.hex("#e03131")
        size = 7.0
    }
    layout {
        title = "ΔCTR per day (Treatment − Control), mean ± 95% CI"
        size = 800 to 300
        xAxisLabel = "ΔCTR (percentage points)"
        yAxisLabel = ""
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="WaX1d2"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 800.0, 
 height: 300.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("WaX1d2");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"ΔCTR per day (Treatment − Control), mean ± 95% CI"
},
"mapping":{
},
"guides":{
"x":{
"title":"ΔCTR (percentage points)"
},
"y":{
"title":""
}
},
"data":{
},
"ggsize":{
"width":800.0,
"height":300.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[0.0,0.0],
"y":[-0.5,0.5]
},
"color":"#adb5bd",
"size":1.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[-0.8020548453847407,-2.076509175301886,6.28100150565698,1.1683963705059128,13.85003518656312,5.961147806886187,13.353761579917709,13.03896320832823,6.611943422342387,-2.576611492759353,4.06686576935653,3.585833812361032,3.979251080011878,1.6071728970152903,-0.9495164965492422,2.711830517028761,6.194483223641849,30.386786279109497,13.055329943580887,-3.533343067088073,6.2766685619914275,5.0260821221942376,13.891921500176046,-0.6362664686148708,0.37373931409742794,1.0527159378589952,10.4700994532834,2.5133143259640516,-0.5252173177793001],
"y":[-0.19157931381937754,0.28346977206498103,-0.16049027881127467,0.3293430475050706,-0.22147731267273355,0.0648346851884421,-0.1439525632518779,-0.2532960391052342,0.06515447764686527,-0.29016311862960154,0.09477812553402132,0.21292874365485925,-0.09395987035557107,-0.12770029321845167,-0.09510440410252513,-0.03941660793500096,0.31853433715773105,-0.03614418602864261,-0.1952089799044308,-0.17384118449651148,0.007651468288318142,0.04953400193160895,0.12640626406661293,0.0665684175198269,0.13426166551990792,0.3028520269556323,-0.20488425173065195,0.2588016865045013,-0.07766427351834526]
},
"color":"#4dabf7",
"size":4.5,
"sampling":"none",
"alpha":0.7,
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[2.60823096723235,8.037136271001744],
"y":[0.0,0.0]
},
"color":"#e03131",
"size":3.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[5.322683619117047],
"y":[0.0]
},
"color":"#e03131",
"size":7.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"14"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, 

In [26]:
// Delta plot: paired differences (Treatment − Control) per day
val purchDiffsPct = DoubleArray(treatmentPurchaseRate.size) { (treatmentPurchaseRate[it] - controlPurchaseRate[it]) * 100 }
val purchDiffMean = purchDiffsPct.mean()
val purchDiffSE = sqrt(purchDiffsPct.variance() / purchDiffsPct.size)
val purchCI95 = 2.048 * purchDiffSE  // t_{0.025, df=28}

val rng2 = kotlin.random.Random(42)

plot {
    // Zero reference line
    line {
        x(listOf(0.0, 0.0))
        y(listOf(-0.5, 0.5))
        color = Color.hex("#adb5bd")
        width = 1.0
    }
    // Individual day differences (jittered strip)
    points {
        x(purchDiffsPct.toList())
        y(List(purchDiffsPct.size) { rng2.nextDouble(-0.35, 0.35) })
        color = Color.hex("#4dabf7")
        size = 4.5
        alpha = 0.7
    }
    // 95% CI bar
    line {
        x(listOf(purchDiffMean - purchCI95, purchDiffMean + purchCI95))
        y(listOf(0.0, 0.0))
        color = Color.hex("#e03131")
        width = 3.0
    }
    // Mean point
    points {
        x(listOf(purchDiffMean))
        y(listOf(0.0))
        color = Color.hex("#e03131")
        size = 7.0
    }
    layout {
        title = "ΔPurchase Rate per day (Treatment − Control), mean ± 95% CI"
        size = 800 to 300
        xAxisLabel = "ΔPurchase Rate (percentage points)"
        yAxisLabel = ""
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="d7dPYt"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 800.0, 
 height: 300.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("d7dPYt");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"ΔPurchase Rate per day (Treatment − Control), mean ± 95% CI"
},
"mapping":{
},
"guides":{
"x":{
"title":"ΔPurchase Rate (percentage points)"
},
"y":{
"title":""
}
},
"data":{
},
"ggsize":{
"width":800.0,
"height":300.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[0.0,0.0],
"y":[-0.5,0.5]
},
"color":"#adb5bd",
"size":1.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[-0.10250777462750592,0.2499926306459334,0.5401870247569871,-0.03314156636308463,0.4428564816112599,1.2974712314989651,0.7720764421655057,1.4437773606118791,-0.33471307362166774,0.3865695408675679,-0.11167088873832902,0.30894044892563544,0.06134435536142482,0.1463782045582179,-0.15761323220630255,0.06272267086364425,0.2551292694888633,1.7710701657186776,1.585907258658491,-0.8884615484542191,0.4280856553511663,0.19481826085925025,-0.0727596557855697,0.21001082740217686,0.15034129070635396,0.11370922835090731,0.8659372034282224,0.6408666649482515,-0.12608252254485836],
"y":[-0.19157931381937754,0.28346977206498103,-0.16049027881127467,0.3293430475050706,-0.22147731267273355,0.0648346851884421,-0.1439525632518779,-0.2532960391052342,0.06515447764686527,-0.29016311862960154,0.09477812553402132,0.21292874365485925,-0.09395987035557107,-0.12770029321845167,-0.09510440410252513,-0.03941660793500096,0.31853433715773105,-0.03614418602864261,-0.1952089799044308,-0.17384118449651148,0.007651468288318142,0.04953400193160895,0.12640626406661293,0.0665684175198269,0.13426166551990792,0.3028520269556323,-0.20488425173065195,0.2588016865045013,-0.07766427351834526]
},
"color":"#4dabf7",
"size":4.5,
"sampling":"none",
"alpha":0.7,
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[0.12310067751433304,0.5735366986537941],
"y":[0.0,0.0]
},
"color":"#e03131",
"size":3.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[0.3483186880840636],
"y":[0.0]
},
"color":"#e03131",
"size":7.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"17"
};


In [27]:
val ctrBoxData = dataFrameOf(
    "value" to (controlCTR.map { it * 100 } + treatmentCTR.map { it * 100 }).toList(),
    "group" to List(controlCTR.size) { "Control" } + List(treatmentCTR.size) { "Treatment" }
)

ctrBoxData.plot {
    boxplot("group", "value") {
        boxes {
            fillColor(Stat.x) {
                scale = categorical("Control" to Color.hex("#4dabf7"), "Treatment" to Color.hex("#ff8787"))
            }
            alpha = 0.5
        }
    }
    points {
        x(ctrBoxData["group"].toList()); y(ctrBoxData["value"].toList())
        color = Color.hex("#495057")
        size = 3.0
        alpha = 0.6
    }
    layout {
        title = "CTR by Group (%)"
        size = 500 to 400
        xAxisLabel = "Group"
        yAxisLabel = "CTR (%)"
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="v56saj"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 500.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("v56saj");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"CTR by Group (%)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Group"
},
"y":{
"title":"CTR (%)"
}
},
"data":{
"x":["Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment"],
"y":[8.48347077458828,6.700264375413087,4.9411210908731995,4.205658772194627,3.6928380211962297,1.8575459285266986,7.983373470128327,5.108297893383443,1.9358294225668233,7.060487474728193,2.5660370887953428,7.9109684116503916,3.1126074025115664,3.685498136926493,7.329460953503382,5.541250041801826,6.68775126323166,3.434980072295857,2.272767345499427,7.62316821603665,6.473676741875975,3.3954419464120726,5.6333172447079916,3.6186178486573337,4.351585706622257,4.720189533617379,4.438279187315572,8.83091199513197,4.184859755987997,7.68141592920354,4.623755200111201,11.222122596530179,5.37405514270054,17.54287320775935,7.818693735412885,21.337135050046037,18.147261101711674,8.54777284490921,4.48387598196884,6.632902858151874,11.496802224011423,7.0918584825234445,5.292671033941784,6.3799444569541395,8.253080558830588,12.882234486873509,33.82176635140535,15.328097289080315,4.089825148948576,12.750345303867402,8.42152406860631,19.52523874488404,2.982351380042463,4.725325020719684,5.772905471476374,14.90837864059897,11.344226321096022,3.6596424382086967]
},
"ggsize":{
"width":500.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"group",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"value",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"fill",
"values":["#4dabf7","#ff8787"],
"limits":["Control","Treatment"]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"ymin":"min",
"lower":"lower",
"middle":"middle",
"upper":"upper",
"ymax":"max",
"fill":"x"
},
"stat":"identity",
"data":{
"middle":[4.720189533617379,8.253080558830588],
"min":[1.8575459285266986,2.982351380042463],
"max":[8.83091199513197,21.337135050046037],
"lower":[3.5267989604765955,5.333363088321162],
"upper":[6.8803759250706396,13.89530656373624],
"x":["Control","Treatment"]
},
"sampling":"none",
"alpha":0.5,
"inherit_aes":false,
"position":"identity",
"geom":"boxplot",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"min"
},{
"type":"float",
"column":"lower"
},{
"type":"float",
"column":"middle"
},{
"type":"float",
"column":"upper"
},{
"type":"float",
"column":"max

In [28]:
val purchaseRateBoxData = dataFrameOf(
    "value" to (controlPurchaseRate.map { it * 100 } + treatmentPurchaseRate.map { it * 100 }).toList(),
    "group" to List(controlPurchaseRate.size) { "Control" } + List(treatmentPurchaseRate.size) { "Treatment" }
)

purchaseRateBoxData.plot {
    boxplot("group", "value") {
        boxes {
            fillColor(Stat.x) {
                scale = categorical("Control" to Color.hex("#4dabf7"), "Treatment" to Color.hex("#ff8787"))
            }
            alpha = 0.5
        }
    }
    points {
        x(purchaseRateBoxData["group"].toList()); y(purchaseRateBoxData["value"].toList())
        color = Color.hex("#495057")
        size = 3.0
        alpha = 0.6
    }
    layout {
        title = "Purchase Rate by Group (%)"
        size = 500 to 400
        xAxisLabel = "Group"
        yAxisLabel = "Purchase Rate (%)"
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="ai92J3"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 500.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("ai92J3");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Purchase Rate by Group (%)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Group"
},
"y":{
"title":"Purchase Rate (%)"
}
},
"data":{
"x":["Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment"],
"y":[0.7472612512393897,0.4221744877726371,0.28243654668175017,0.46653311012925713,0.7004290586380139,0.35110432512682677,0.5080328571899845,0.41291662545742264,0.624022308372441,0.41215823405381485,0.680732859506683,0.9245959419170278,0.5425203789380921,0.2755090519025932,0.6145298425793416,0.18560010701267432,0.2240622579574374,0.5023635183983687,0.2635986952305387,1.071610362472205,0.36611323967645804,0.5897751770865415,0.8067296421430049,0.32989761208826895,0.20096559565081626,0.5779628957374042,0.45392293899219227,0.3629290767040824,0.6019441898909312,0.6447534766118838,0.6721671184185705,0.8226235714387372,0.4333915437661725,1.1432855402492736,1.648575556625792,1.2801092993554901,1.8566939860693017,0.28930923475077325,0.7987277749213827,0.5690619707683541,1.2335363908426633,0.6038647342995169,0.42188725646081116,0.4569166103730391,0.24832277787631857,0.47919152744630067,2.2734336841170464,1.8495059538890297,0.18314881401798594,0.7941988950276244,0.7845934379457917,0.7339699863574352,0.5399084394904459,0.3513068863571702,0.6916721240883116,1.3198601424204148,1.0037957416523338,0.47586166734607294]
},
"ggsize":{
"width":500.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"group",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"value",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"fill",
"values":["#4dabf7","#ff8787"],
"limits":["Control","Treatment"]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"ymin":"min",
"lower":"lower",
"middle":"middle",
"upper":"upper",
"ymax":"max",
"fill":"x"
},
"stat":"identity",
"data":{
"min":[0.18560010701267432,0.18314881401798594],
"middle":[0.46653311012925713,0.6916721240883116],
"max":[0.9245959419170278,1.8566939860693017],
"lower":[0.34050096860754786,0.466389138859556],
"upper":[0.6192760754758913,1.1884109655459685],
"x":["Control","Treatment"]
},
"sampling":"none",
"alpha":0.5,
"inherit_aes":false,
"position":"identity",
"geom":"boxplot",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"min"
},{
"type":"float",
"column":"lower"
},{
"type":"float",


The histograms show the distribution shape with rug marks for individual observations.
With ~29 points per group, histograms avoid the over-smoothing problem of KDE — bin edges
are visible rather than artificially smooth curves. The boxplots with overlaid points give
a complementary view of medians, quartiles, and outliers.

### 10. Assumption Checks

For paired tests, we check the **differences** (treatment − control per day):
1. **Normality of differences** (Shapiro-Wilk) — are the paired differences approximately normal?
2. **Homogeneity of variance** (Levene) — reported for completeness, though paired tests do not require it.

In [29]:
val ctrDiff = treatmentCTR.zip(controlCTR).map { (t, c) -> t - c }.toDoubleArray()
val ctrDiffNorm = shapiroWilkTest(ctrDiff)

println("CTR differences (treatment - control):")
println("  mean Δ = ${(ctrDiff.describe().mean * 100).fmt(2)} pp")
println("  Shapiro-Wilk: p=${ctrDiffNorm.pValue.fmt(4)} ${if (ctrDiffNorm.pValue > 0.05) "✓ normal" else "✗ non-normal"}")

// Also show per-group normality for reference
val ctrNormCtrl = shapiroWilkTest(controlCTR)
val ctrNormTreat = shapiroWilkTest(treatmentCTR)
println("  Per-group — Control: p=${ctrNormCtrl.pValue.fmt(4)}, Treatment: p=${ctrNormTreat.pValue.fmt(4)}")

CTR differences (treatment - control):
  mean Δ = 5.32 pp
  Shapiro-Wilk: p=0.0012 ✗ non-normal
  Per-group — Control: p=0.2448, Treatment: p=0.0007


In [30]:
val purchRateDiff = treatmentPurchaseRate.zip(controlPurchaseRate).map { (t, c) -> t - c }.toDoubleArray()
val purchRateDiffNorm = shapiroWilkTest(purchRateDiff)

println("Purchase Rate differences (treatment - control):")
println("  mean Δ = ${(purchRateDiff.describe().mean * 100).fmt(3)} pp")
println("  Shapiro-Wilk: p=${purchRateDiffNorm.pValue.fmt(4)} ${if (purchRateDiffNorm.pValue > 0.05) "✓ normal" else "✗ non-normal"}")

// Also show per-group normality for reference
val purchRateNormCtrl = shapiroWilkTest(controlPurchaseRate)
val purchRateNormTreat = shapiroWilkTest(treatmentPurchaseRate)
println("  Per-group — Control: p=${purchRateNormCtrl.pValue.fmt(4)}, Treatment: p=${purchRateNormTreat.pValue.fmt(4)}")

Purchase Rate differences (treatment - control):
  mean Δ = 0.348 pp
  Shapiro-Wilk: p=0.0309 ✗ non-normal
  Per-group — Control: p=0.2822, Treatment: p=0.0053


**Decision tree for paired data**:
- Shapiro-Wilk on **differences** passes (p > 0.05) → **paired t-test** is primary
- Shapiro-Wilk fails → paired t-test is often robust to mild non-normality, but at N≈29
  results should be interpreted cautiously. The **Wilcoxon signed-rank** serves as a
  sensitivity check — if both tests agree, the conclusion is more trustworthy
- No need to check equal variances — paired tests work on differences, not separate groups

We report both parametric (paired t-test) and non-parametric (Wilcoxon signed-rank) tests.

### 11. Hypothesis Tests — Paired t-test & Wilcoxon Signed-Rank (on rate metrics)

In [31]:
val ctrT = pairedTTest(treatmentCTR, controlCTR)
val ctrW = wilcoxonSignedRankTest(treatmentCTR, controlCTR)

println("CTR — Paired t: t=${ctrT.statistic.fmt(3)}, p=${ctrT.pValue.fmt(4)}, CI=[${(ctrT.confidenceInterval!!.lower * 100).fmt(2)}%, ${(ctrT.confidenceInterval!!.upper * 100).fmt(2)}%]")
println("CTR — Significant: ${ctrT.isSignificant()}")
println()
println("CTR — Wilcoxon signed-rank: W=${ctrW.statistic.fmt(1)}, p=${ctrW.pValue.fmt(4)}")
"CTR — Significant: ${ctrW.isSignificant()}"

CTR — Paired t: t=4.016, p=0.0004, CI=[2.61%, 8.04%]
CTR — Significant: true

CTR — Wilcoxon signed-rank: W=388.0, p=0.0002


CTR — Significant: true

In [32]:
val purchRateT = pairedTTest(treatmentPurchaseRate, controlPurchaseRate)
val purchRateW = wilcoxonSignedRankTest(treatmentPurchaseRate, controlPurchaseRate)

println("Purchase Rate — Paired t: t=${purchRateT.statistic.fmt(3)}, p=${purchRateT.pValue.fmt(4)}, CI=[${(purchRateT.confidenceInterval!!.lower * 100).fmt(3)}%, ${(purchRateT.confidenceInterval!!.upper * 100).fmt(3)}%]")
println("Purchase Rate — Significant: ${purchRateT.isSignificant()}")
println()
println("Purchase Rate — Wilcoxon signed-rank: W=${purchRateW.statistic.fmt(1)}, p=${purchRateW.pValue.fmt(4)}")
"Purchase Rate — Significant: ${purchRateW.isSignificant()}"

Purchase Rate — Paired t: t=3.167, p=0.0037, CI=[0.123%, 0.574%]
Purchase Rate — Significant: true

Purchase Rate — Wilcoxon signed-rank: W=358.0, p=0.0025


Purchase Rate — Significant: true

In [33]:
// Strip plot with mean ± SE: shows every observation
val ctrStripGroups = List(controlCTR.size) { "Control" } + List(treatmentCTR.size) { "Treatment" }
val ctrStripValues = (controlCTR.map { it * 100 } + treatmentCTR.map { it * 100 }).toList()

plot {
    points {
        x(ctrStripGroups); y(ctrStripValues)
        color = Color.hex("#868e96")
        size = 4.0
        alpha = 0.5
        position = Position.jitter(width = 0.15)
    }
    points {
        x(listOf("Control", "Treatment"))
        y(listOf(ctrlCTRDesc.mean * 100, treatCTRDesc.mean * 100))
        color = Color.hex("#212529")
        size = 7.0
    }
    errorBars {
        x(listOf("Control", "Treatment"))
        yMin(listOf(
            (ctrlCTRDesc.mean - ctrlCTRDesc.standardError) * 100,
            (treatCTRDesc.mean - treatCTRDesc.standardError) * 100
        ))
        yMax(listOf(
            (ctrlCTRDesc.mean + ctrlCTRDesc.standardError) * 100,
            (treatCTRDesc.mean + treatCTRDesc.standardError) * 100
        ))
        width = 0.2
    }
    layout {
        title = "CTR: Strip + Mean ± SE"
        size = 500 to 400
        xAxisLabel = "Group"
        yAxisLabel = "CTR (%)"
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="4iz20h"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 500.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("4iz20h");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"CTR: Strip + Mean ± SE"
},
"mapping":{
},
"guides":{
"x":{
"title":"Group"
},
"y":{
"title":"CTR (%)"
}
},
"data":{
},
"ggsize":{
"width":500.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment"],
"y":[8.48347077458828,6.700264375413087,4.9411210908731995,4.205658772194627,3.6928380211962297,1.8575459285266986,7.983373470128327,5.108297893383443,1.9358294225668233,7.060487474728193,2.5660370887953428,7.9109684116503916,3.1126074025115664,3.685498136926493,7.329460953503382,5.541250041801826,6.68775126323166,3.434980072295857,2.272767345499427,7.62316821603665,6.473676741875975,3.3954419464120726,5.6333172447079916,3.6186178486573337,4.351585706622257,4.720189533617379,4.438279187315572,8.83091199513197,4.184859755987997,7.68141592920354,4.623755200111201,11.222122596530179,5.37405514270054,17.54287320775935,7.818693735412885,21.337135050046037,18.147261101711674,8.54777284490921,4.48387598196884,6.632902858151874,11.496802224011423,7.0918584825234445,5.292671033941784,6.3799444569541395,8.253080558830588,12.882234486873509,33.82176635140535,15.328097289080315,4.089825148948576,12.750345303867402,8.42152406860631,19.52523874488404,2.982351380042463,4.725325020719684,5.772905471476374,14.90837864059897,11.344226321096022,3.6596424382086967]
},
"color":"#868e96",
"size":4.0,
"sampling":"none",
"alpha":0.5,
"inherit_aes":false,
"position":{
"name":"jitter",
"width":0.15
},
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["Control","Treatment"],
"y":[5.095870900557932,10.41855451967498]
},
"color":"#212529",
"size":7.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[4.715052630372424,9.152054100461957],
"ymax":[5.47668917074344,11.685054938888003],
"x":["Control","Treatment"]
},
"sampling":"none",
"width":0.2,
"inherit_aes":false,
"position":"dodge",
"geo

In [34]:
val purchRateStripGroups = List(controlPurchaseRate.size) { "Control" } + List(treatmentPurchaseRate.size) { "Treatment" }
val purchRateStripValues = (controlPurchaseRate.map { it * 100 } + treatmentPurchaseRate.map { it * 100 }).toList()

plot {
    points {
        x(purchRateStripGroups); y(purchRateStripValues)
        color = Color.hex("#868e96")
        size = 4.0
        alpha = 0.5
        position = Position.jitter(width = 0.15)
    }
    points {
        x(listOf("Control", "Treatment"))
        y(listOf(ctrlPurchRateDesc.mean * 100, treatPurchRateDesc.mean * 100))
        color = Color.hex("#212529")
        size = 7.0
    }
    errorBars {
        x(listOf("Control", "Treatment"))
        yMin(listOf(
            (ctrlPurchRateDesc.mean - ctrlPurchRateDesc.standardError) * 100,
            (treatPurchRateDesc.mean - treatPurchRateDesc.standardError) * 100
        ))
        yMax(listOf(
            (ctrlPurchRateDesc.mean + ctrlPurchRateDesc.standardError) * 100,
            (treatPurchRateDesc.mean + treatPurchRateDesc.standardError) * 100
        ))
        width = 0.2
    }
    layout {
        title = "Purchase Rate: Strip + Mean ± SE"
        size = 500 to 400
        xAxisLabel = "Group"
        yAxisLabel = "Purchase Rate (%)"
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="L9V1lW"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 500.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("L9V1lW");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Purchase Rate: Strip + Mean ± SE"
},
"mapping":{
},
"guides":{
"x":{
"title":"Group"
},
"y":{
"title":"Purchase Rate (%)"
}
},
"data":{
},
"ggsize":{
"width":500.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Control","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment","Treatment"],
"y":[0.7472612512393897,0.4221744877726371,0.28243654668175017,0.46653311012925713,0.7004290586380139,0.35110432512682677,0.5080328571899845,0.41291662545742264,0.624022308372441,0.41215823405381485,0.680732859506683,0.9245959419170278,0.5425203789380921,0.2755090519025932,0.6145298425793416,0.18560010701267432,0.2240622579574374,0.5023635183983687,0.2635986952305387,1.071610362472205,0.36611323967645804,0.5897751770865415,0.8067296421430049,0.32989761208826895,0.20096559565081626,0.5779628957374042,0.45392293899219227,0.3629290767040824,0.6019441898909312,0.6447534766118838,0.6721671184185705,0.8226235714387372,0.4333915437661725,1.1432855402492736,1.648575556625792,1.2801092993554901,1.8566939860693017,0.28930923475077325,0.7987277749213827,0.5690619707683541,1.2335363908426633,0.6038647342995169,0.42188725646081116,0.4569166103730391,0.24832277787631857,0.47919152744630067,2.2734336841170464,1.8495059538890297,0.18314881401798594,0.7941988950276244,0.7845934379457917,0.7339699863574352,0.5399084394904459,0.3513068863571702,0.6916721240883116,1.3198601424204148,1.0037957416523338,0.47586166734607294]
},
"color":"#868e96",
"size":4.0,
"sampling":"none",
"alpha":0.5,
"inherit_aes":false,
"position":{
"name":"jitter",
"width":0.15
},
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["Control","Treatment"],
"y":[0.5000838685705585,0.8484025566546223]
},
"color":"#212529",
"size":7.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[0.4597583913982098,0.7500408688430775],
"ymax":[0.5404093457429073,0.9467642444661669],
"x":["Control","Treatmen

Paired tests account for day-level variability (weekday effects, seasonality) by analyzing
within-day differences. When the paired t-test and Wilcoxon signed-rank agree, confidence
is higher. When they disagree, check for outliers in the paired differences.

The strip plots overlay individual observations on top of mean ± SE. Note that SE bars reflect
*between-day* variability, while the paired tests use *within-day* differences — do not judge
significance from visual overlap of error bars.

### 12. Effect Size — Paired Cohen's dz

| dz | Interpretation |
|----|----------------|
| < 0.2 | Negligible |
| 0.2 | Small |
| 0.5 | Medium |
| 0.8 | Large |

For paired designs, Cohen's dz = mean(differences) / SD(differences) is the appropriate
effect size. It captures how large the within-pair differences are relative to their variability.
The same conventional thresholds (0.2, 0.5, 0.8) apply.

In [35]:
fun interpretD(d: Double) = when {
    abs(d) < 0.2 -> "negligible"
    abs(d) < 0.5 -> "small"
    abs(d) < 0.8 -> "medium"
    else -> "large"
}

// Paired dz: mean of within-day differences / SD of those differences
val ctrDiffs = DoubleArray(treatmentCTR.size) { treatmentCTR[it] - controlCTR[it] }
val purchDiffs = DoubleArray(treatmentPurchaseRate.size) { treatmentPurchaseRate[it] - controlPurchaseRate[it] }

val ctrDz = ctrDiffs.mean() / sqrt(ctrDiffs.variance())
val purchDz = purchDiffs.mean() / sqrt(purchDiffs.variance())

println("CTR           — paired dz = ${ctrDz.fmt(3)} (${interpretD(ctrDz)})")
"Purchase Rate — paired dz = ${purchDz.fmt(3)} (${interpretD(purchDz)})"

CTR           — paired dz = 0.746 (medium)


Purchase Rate — paired dz = 0.588 (medium)

Cohen's dz is the paired effect size: the mean of within-day differences divided by their
standard deviation. Unlike unpaired Cohen's d (which uses pooled between-group SD), dz
directly reflects how consistently one condition outperforms the other across matched pairs.
Larger dz values indicate more consistent day-over-day advantages for the treatment group.

### 13. Multiple Comparison Correction

We tested two metrics: CTR and Purchase Rate. Testing multiple hypotheses inflates the chance of
false positives (Type I error). Three correction methods:

| Method | Controls | Conservatism |
|--------|----------|-------------|
| **Bonferroni** | FWER (probability of *any* false positive) | Most conservative |
| **Holm-Bonferroni** | FWER | Less conservative, uniformly more powerful |
| **Benjamini-Hochberg** | FDR (expected *proportion* of false positives) | Least conservative |

Use **FWER** (Bonferroni/Holm) when any false positive is costly (e.g., regulatory).
Use **FDR** (BH) when screening many metrics and can tolerate some false positives.

In [36]:
val rawPValues = doubleArrayOf(ctrT.pValue, purchRateT.pValue)
val metricNames = listOf("CTR", "Purch.Rate")

val bonf = bonferroniCorrection(rawPValues)
val holm = holmBonferroniCorrection(rawPValues)
val bh   = benjaminiHochbergCorrection(rawPValues)

println("%-10s  %-10s  %-12s  %-12s  %-10s".format("Metric", "Raw p", "Bonferroni", "Holm", "BH"))
println("-".repeat(60))
for (i in metricNames.indices) {
    println("%-10s  %-10s  %-12s  %-12s  %-10s".format(
        metricNames[i], rawPValues[i].fmt(4), bonf[i].fmt(4), holm[i].fmt(4), bh[i].fmt(4)
    ))
}

Metric      Raw p       Bonferroni    Holm          BH        
------------------------------------------------------------
CTR         0.0004      0.0008        0.0008        0.0008    
Purch.Rate  0.0037      0.0074        0.0037        0.0037    


In [37]:
val allPLabels = metricNames.flatMap { m -> listOf("$m\n(raw)", "$m\n(Bonf)", "$m\n(Holm)", "$m\n(BH)") }
val allPValues = (0 until 2).flatMap { i -> listOf(rawPValues[i], bonf[i], holm[i], bh[i]) }
val allPType = (0 until 2).flatMap { listOf("Raw", "Bonferroni", "Holm", "BH") }

// -log10 scale: taller bars = more significant
val negLog10 = allPValues.map { -log10(it) }
val alphaLine = -log10(0.05)  // ≈ 1.30

plot {
    bars {
        x(allPLabels); y(negLog10)
        fillColor(allPType)
        alpha = 0.8
    }
    line {
        x(listOf(allPLabels.first(), allPLabels.last()))
        y(listOf(alphaLine, alphaLine))
        color = Color.hex("#e03131")
        width = 1.5
    }
    layout {
        title = "Raw vs Adjusted p-values (red line = α = 0.05)"
        size = 900 to 400
        xAxisLabel = "Metric (correction)"
        yAxisLabel = "-log10(p-value)"
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="AC5yqk"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 900.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("AC5yqk");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Raw vs Adjusted p-values (red line = α = 0.05)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Metric (correction)"
},
"y":{
"title":"-log10(p-value)"
}
},
"data":{
},
"ggsize":{
"width":900.0,
"height":400.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"fill":"fill"
},
"stat":"identity",
"data":{
"x":["CTR\n(raw)","CTR\n(Bonf)","CTR\n(Holm)","CTR\n(BH)","Purch.Rate\n(raw)","Purch.Rate\n(Bonf)","Purch.Rate\n(Holm)","Purch.Rate\n(BH)"],
"y":[3.3950087105781437,3.0939787149141624,3.0939787149141624,3.0939787149141624,2.4321059791178947,2.1310759834539135,2.4321059791178947,2.4321059791178947],
"fill":["Raw","Bonferroni","Holm","BH","Raw","Bonferroni","Holm","BH"]
},
"sampling":"none",
"alpha":0.8,
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
},{
"type":"str",
"column":"fill"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["CTR\n(raw)","Purch.Rate\n(BH)"],
"y":[1.3010299956639813,1.3010299956639813]
},
"color":"#e03131",
"size":1.5,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"32"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CTR 
 
 
 (raw) 
 
 
 
 
 
 
 
 
 CTR 
 
 
 (Bonf) 
 
 
 
 
 
 
 
 
 CTR 
 
 
 (Holm) 
 
 
 
 
 
 
 
 
 CTR 
 
 
 (BH) 
 
 
 
 
 
 
 
 
 Purch.Rate 
 
 
 (raw) 
 
 
 
 
 
 
 
 
 Purch.Rate 
 
 
 (Bonf) 
 
 
 
 
 
 
 
 
 Purch.Rate 
 
 
 (Holm) 
 
 
 
 
 
 
 
 
 Purch.Rate 
 
 
 (BH) 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 0.5 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 1.5 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 2.5 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 3.5 
 
 
 
 
 
 
 
 
 Raw vs Adjusted p-values (red line = α = 0.05) 
 
 
 
 
 -log10(p-value) 
 
 
 
 
 Metric (

After correction, Holm is generally recommended as the default: it controls FWER like
Bonferroni but is strictly more powerful.

### 14. Metric Correlation — CTR vs Purchase Rate

Are our two test metrics correlated? Bonferroni controls FWER regardless of dependence
structure, but when metrics are positively correlated the correction becomes more
conservative than necessary (the actual FWER is below the nominal α).

In [38]:
val allCTR = controlCTR + treatmentCTR
val allPurchaseRate = controlPurchaseRate + treatmentPurchaseRate

val corrP = pearsonCorrelation(allCTR, allPurchaseRate)
val corrS = spearmanCorrelation(allCTR, allPurchaseRate)

println("Pearson  r = ${corrP.coefficient.fmt(3)}, p = ${"%.2e".format(corrP.pValue)}")
"Spearman ρ = ${corrS.coefficient.fmt(3)}, p = ${"%.2e".format(corrS.pValue)}"

Pearson  r = 0.748, p = 1,55e-11


Spearman ρ = 0.517, p = 3,22e-05

In [39]:
plot {
    points {
        x(allCTR.map { it * 100 }.toList()); y(allPurchaseRate.map { it * 100 }.toList())
        color = Color.hex("#9775fa")
        size = 5.0
        alpha = 0.6
    }
    layout {
        title = "CTR vs Purchase Rate (all campaign data, r=${corrP.coefficient.fmt(2)})"
        size = 700 to 500
        xAxisLabel = "CTR (%)"
        yAxisLabel = "Purchase Rate (%)"
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="MQXeKp"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 500.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("MQXeKp");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"CTR vs Purchase Rate (all campaign data, r=0.75)"
},
"mapping":{
},
"guides":{
"x":{
"title":"CTR (%)"
},
"y":{
"title":"Purchase Rate (%)"
}
},
"data":{
},
"ggsize":{
"width":700.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":[8.48347077458828,6.700264375413087,4.9411210908731995,4.205658772194627,3.6928380211962297,1.8575459285266986,7.983373470128327,5.108297893383443,1.9358294225668233,7.060487474728193,2.5660370887953428,7.9109684116503916,3.1126074025115664,3.685498136926493,7.329460953503382,5.541250041801826,6.68775126323166,3.434980072295857,2.272767345499427,7.62316821603665,6.473676741875975,3.3954419464120726,5.6333172447079916,3.6186178486573337,4.351585706622257,4.720189533617379,4.438279187315572,8.83091199513197,4.184859755987997,7.68141592920354,4.623755200111201,11.222122596530179,5.37405514270054,17.54287320775935,7.818693735412885,21.337135050046037,18.147261101711674,8.54777284490921,4.48387598196884,6.632902858151874,11.496802224011423,7.0918584825234445,5.292671033941784,6.3799444569541395,8.253080558830588,12.882234486873509,33.82176635140535,15.328097289080315,4.089825148948576,12.750345303867402,8.42152406860631,19.52523874488404,2.982351380042463,4.725325020719684,5.772905471476374,14.90837864059897,11.344226321096022,3.6596424382086967],
"y":[0.7472612512393897,0.4221744877726371,0.28243654668175017,0.46653311012925713,0.7004290586380139,0.35110432512682677,0.5080328571899845,0.41291662545742264,0.624022308372441,0.41215823405381485,0.680732859506683,0.9245959419170278,0.5425203789380921,0.2755090519025932,0.6145298425793416,0.18560010701267432,0.2240622579574374,0.5023635183983687,0.2635986952305387,1.071610362472205,0.36611323967645804,0.5897751770865415,0.8067296421430049,0.32989761208826895,0.20096559565081626,0.5779628957374042,0.45392293899219227,0.3629290767040824,0.6019441898909312,0.6447534766118838,0.6721671184185705,0.8226235714387372,0.4333915437661725,1.1432855402492736,1.648575556625792,1.2801092993554901,1.8566939860693017,0.28930923475077325,0.7987277749213827,0.5690619707683541,1.2335363908426633,0.6038647342995169,0.42188725646081116,0.4569166103730391,0.24832277787631857,0.47919152744630067,2.2734336841170464,1.8495059538890297,0.18314881401798594,0.7941988950276244,0.7845934379457917,0.7339699863574352,0.5399084394904459,0.3513068863571702,0.6916721240883116,1.3198601424204148,1.0037957416523338,0.47586166734607294]
},
"color":"#9775fa",
"size":5.0,
"sampling":"none",
"alpha":0.6,
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"35"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === '

In [40]:
// Correlation heatmap: off-diagonal only, scale centered at 0
val allSpend = (dfControlPaired["Spend [USD]"].toList() + dfTestPaired["Spend [USD]"].toList()).map { (it as Number).toDouble() }.toDoubleArray()
val allImpressions = (dfControlPaired["# of Impressions"].toList() + dfTestPaired["# of Impressions"].toList()).map { (it as Number).toDouble() }.toDoubleArray()

val metricsData = listOf(allSpend, allImpressions, allCTR, allPurchaseRate)
val corrMetricNames = listOf("Spend", "Impressions", "CTR", "Purchase Rate")

val heatX = mutableListOf<String>()
val heatY = mutableListOf<String>()
val heatR = mutableListOf<Double>()
val heatLabel = mutableListOf<String>()
for (i in corrMetricNames.indices) {
    for (j in corrMetricNames.indices) {
        if (i == j) continue  // skip diagonal
        heatX.add(corrMetricNames[j]); heatY.add(corrMetricNames[i])
        val r = pearsonCorrelation(metricsData[i], metricsData[j]).coefficient
        heatR.add(r)
        heatLabel.add(r.fmt(2))
    }
}

plot {
    tiles {
        x(heatX); y(heatY)
        fillColor(heatR) {
            scale = continuousColorBrewer(BrewerPalette.Diverging.RdBu, domain = -1.0..1.0)
        }
    }
    text {
        x(heatX); y(heatY)
        label(heatLabel)
        font { size = 12.0 }
    }
    layout {
        title = "Correlation Matrix — Rate Metrics & Exposure (off-diagonal)"
        size = 650 to 550
        xAxisLabel = "Metric"
        yAxisLabel = "Metric"
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="vw7QMp"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 650.0, 
 height: 550.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("vw7QMp");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Correlation Matrix — Rate Metrics & Exposure (off-diagonal)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Metric"
},
"y":{
"title":"Metric"
}
},
"data":{
},
"ggsize":{
"width":650.0,
"height":550.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"discrete":true
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_brewer",
"palette":"RdBu",
"limits":[-1.0,1.0]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y",
"fill":"fill"
},
"stat":"identity",
"data":{
"x":["Impressions","CTR","Purchase Rate","Spend","CTR","Purchase Rate","Spend","Impressions","Purchase Rate","Spend","Impressions","CTR"],
"y":["Spend","Spend","Spend","Impressions","Impressions","Impressions","CTR","CTR","CTR","Purchase Rate","Purchase Rate","Purchase Rate"],
"fill":[-0.05749260068031413,0.07626419705629771,0.1408425225239475,-0.05749260068031413,-0.7760717631610394,-0.7010996070190462,0.07626419705629771,-0.7760717631610394,0.7476636007600962,0.1408425225239475,-0.7010996070190462,0.7476636007600962]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"tile",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"str",
"column":"y"
},{
"type":"float",
"column":"fill"
}]
}
},{
"mapping":{
"x":"x",
"y":"y",
"label":"label"
},
"stat":"identity",
"data":{
"x":["Impressions","CTR","Purchase Rate","Spend","CTR","Purchase Rate","Spend","Impressions","Purchase Rate","Spend","Impressions","CTR"],
"y":["Spend","Spend","Spend","Impressions","Impressions","Impressions","CTR","CTR","CTR","Purchase Rate","Purchase Rate","Purchase Rate"],
"label":["-0.06","0.08","0.14","-0.06","-0.78","-0.70","0.08","-0.78","0.75","0.14","-0.70","0.75"]
},
"size":12.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"text",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"str",
"column":"y"
},{
"type":"str",
"column":"label"
}]
}
}],
"spec_id":"38"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.06 
 
 
 
 
 
 
 0.08 
 
 
 
 
 
 
 0.14 
 
 
 
 
 
 
 -0.06 
 
 
 
 
 
 
 -0.78 
 
 
 
 
 
 
 -0.70 
 

Positively correlated metrics mean the Bonferroni correction is more conservative than
necessary — the actual FWER is below the nominal level. Bonferroni remains valid (it never
*loses* FWER control), but it sacrifices power. For highly correlated metrics,
permutation-based or resampling-based corrections can recover some of that lost power.

---

## Summary

| Metric | Test | p-value | Effect Size | Decision |
|--------|------|---------|-------------|----------|
| Conversion rate | Proportion z-test | ~0.19 | Cohen's h ≈ -0.005 (negligible) | No difference (power >99% for 1 pp lift) |
| Conversion rate | Chi-squared | ~0.19 | — | Confirms z-test |
| Conversion rate | Bayesian P(B>A) | — | — | ~9.5% probability treatment is better |
| CTR (clicks/impressions) | Paired t-test | ~0.0004 (Holm: 0.0008) | Paired dz ≈ 0.75 (medium–large) | Significant — treatment has higher CTR |
| Purchase rate (purchases/impressions) | Paired t-test | ~0.0037 (Holm: 0.0037) | Paired dz ≈ 0.59 (medium) | Significant — treatment has higher purchase rate |

**Note on Act 2 methodology**: The marketing campaign groups had very different exposure levels
(treatment received ~30% fewer impressions but ~12% more spend). Raw daily counts are not
directly comparable. All Act 2 analyses use **rate metrics** (CTR and purchase rate per
impression) to remove this confound. Since both campaigns ran on the same calendar days, we
use **paired tests** (paired t-test, Wilcoxon signed-rank) matched by date to control for
day-level variability.

**Caveat on Act 2 power**: With only ~29 matched pairs, the study has moderate power (~55% for
a medium effect dz ≈ 0.4). The significant results we observed correspond to medium-to-large
effects (dz ≈ 0.59–0.75), which are well above the detectable threshold. Nevertheless, the wide
confidence intervals reflect the small sample — point estimates of the effect size carry
considerable uncertainty.

**Business conclusion**: The new landing page does not improve conversion rate — the experiment was
well-powered (>99% to detect a 1 pp lift) and both frequentist and Bayesian analyses agree.
For the marketing campaign, both CTR and purchase rate are significantly higher in the treatment
group after Holm correction for multiple comparisons. However, with only 29 matched days the
effect size estimates are imprecise; a longer experiment would narrow the confidence intervals
and confirm the magnitude of the improvement.